# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.date_published}")

## 2. Data Overview

Review available record sets, fields, columns, and their @id references.

In [ ]:
# List record sets with their @id and name

print("Available Record Sets:")
record_sets = []
for record_set in dataset.record_sets:
    print(f"  @id: {record_set.id}\n    name: {record_set.name}\n    description: {getattr(record_set, 'description', '')}\n")
    record_sets.append(record_set)

# For each record set, list its fields (with @id and name)
for record_set in record_sets:
    print(f"Record set: {record_set.name} (@id: {record_set.id})")
    for field in record_set.fields:
        print(f"    Field: {field.name} (@id: {field.id}) - type: {getattr(field, 'data_type', 'unknown')}")
    print("")

## 3. Data Extraction

Load records from a specific record set into a pandas DataFrame for analysis. Use the record set and field @id values identified above.


In [ ]:
# Get the @id for each record set
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(records)
    dataframes[rec_id] = df
    print(f"Loaded record set @id: {rec_id} (rows: {df.shape[0]})")

if record_set_ids:
    print("\nColumns in the first record set:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    print("\nSample records:")
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)

Apply some basic EDA: filter on a numeric field, perform normalization, and show groupby results (if applicable). All field references by @id.


In [ ]:
# Select a record set and a numeric field for demonstration

# Use the first record set as an example (if exists)
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    
    # List all fields with their @id
    rs = None
    for rs_object in dataset.record_sets:
        if rs_object.id == rs_id:
            rs = rs_object
            break
    numeric_fields = [field for field in rs.fields if getattr(field, 'data_type', '').lower() in ("number","float","integer")]
    if numeric_fields:
        numeric_field = numeric_fields[0].id
        print(f"Using numeric field for EDA: {numeric_field}")
        # Ensure conversion to numeric in case data is string
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {filtered_df.shape[0]} rows")
        display(filtered_df[[numeric_field]].head())
        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field}:")
        display(filtered_df[[numeric_field, norm_col]].head())
        
        # Try to group by another field if one exists
        categorical_fields = [field for field in rs.fields if getattr(field, 'data_type', '').lower() == "text"]
        if categorical_fields:
            group_field = categorical_fields[0].id
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"\nGrouped mean of {numeric_field} by {group_field}:")
                display(grouped_df.head())
    else:
        print("No numeric field found for EDA.")

## 5. Visualization

Visualize data distributions and relationships between fields. (Optional depending on data availability.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Continue only if at least one numeric field in first record set
if record_set_ids and 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a categorical field exists, plot mean by category
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load a FAIR^2-compliant dataset using `mlcroissant`, list all record sets and their fields, extract data by record set @id, conduct basic exploratory analysis, and visualize distribution of numeric variables. All references to entities are via their unique `@id` as per best practice. For further analysis, consult the Croissant schema and consider examining all available record sets and fields in depth.